In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/lizhecheng/mushroom-classification/Transfer data/Transfer data/Exidia/JAH2014PIC47794713.jpg
/kaggle/input/datasets/lizhecheng/mushroom-classification/Transfer data/Transfer data/Exidia/RSS2017-9188091_H1ey3FQyoe.jpg
/kaggle/input/datasets/lizhecheng/mushroom-classification/Transfer data/Transfer data/Exidia/MSO2011PIC32526200.jpg
/kaggle/input/datasets/lizhecheng/mushroom-classification/Transfer data/Transfer data/Exidia/APE2016-9182993_S1biMobGGe.jpg
/kaggle/input/datasets/lizhecheng/mushroom-classification/Transfer data/Transfer data/Exidia/JM2017-9190112_SygybwWApe.jpg
/kaggle/input/datasets/lizhecheng/mushroom-classification/Transfer data/Transfer data/Exidia/PTP2015PIC52641717.jpg
/kaggle/input/datasets/lizhecheng/mushroom-classification/Transfer data/Transfer data/Exidia/MFC2016-9180603_HJlPfl0lxe.jpg
/kaggle/input/datasets/lizhecheng/mushroom-classification/Transfer data/Transfer data/Exidia/MC2017-9188766_ryeCq1Q3l.jpg
/kaggle/input/datasets/lizhecheng/m

In [2]:
import os

BASE_PATH = "/kaggle/input/datasets/lizhecheng/mushroom-classification"

print("=" * 60)
print("DATASET STRUCTURE")
print("=" * 60)

for root, dirs, files in os.walk(BASE_PATH):

    level = root.replace(BASE_PATH, "").count(os.sep)
    indent = "    " * level

    print(f"{indent}{os.path.basename(root)}/")

    for file in files[:5]:
        print(f"{indent}    {file}")

    if level >= 3:
        dirs[:] = []

DATASET STRUCTURE
mushroom-classification/
    Transfer data/
        Transfer data/
            Exidia/
                JAH2014PIC47794713.jpg
                RSS2017-9188091_H1ey3FQyoe.jpg
                MSO2011PIC32526200.jpg
                APE2016-9182993_S1biMobGGe.jpg
                JM2017-9190112_SygybwWApe.jpg
            Inocybe/
                TRL2017-9205078_BkEt7rOdZ.jpg
                JCS2013PIC50081502.jpg
                TB2014PIC25857679.jpg
                TOB2017-9208818_BySIHxHg5-.jpg
                TRL2017-9205078_ryf3VKTOb.jpg
    archive/
        Mushrooms/
            Entoloma/
                126_-P40YY9dxnU.jpg
                092_C-d5aHykeOY.jpg
                033_yJBfhGv9URU.jpg
                154_HDT_he4bSpA.jpg
                159_gzyo4cu2SF0.jpg
            Suillus/
                168_nQy5xpC7tWk.jpg
                000_o5wtwaqK-UE.jpg
                317_41mNOdPZlRo.jpg
                244_wFrkfThPI18.jpg
                139_I8Oaix9-ru4.jpg
     

In [3]:
import os
from collections import Counter

BASE_PATH = "/kaggle/input/datasets/lizhecheng/mushroom-classification"

DATASETS = {
    "Transfer_Data": os.path.join(
        BASE_PATH, "Transfer data", "Transfer data"
    ),
    
    "Archive_Mushrooms": os.path.join(
        BASE_PATH, "archive", "Mushrooms"
    )
}

for dataset_name, dataset_path in DATASETS.items():

    print("\n" + "=" * 70)
    print(dataset_name)
    print("=" * 70)
    print("Path:", dataset_path)

    if not os.path.exists(dataset_path):
        print("❌ Path not found")
        continue

    classes = sorted([
        folder for folder in os.listdir(dataset_path)
        if os.path.isdir(os.path.join(dataset_path, folder))
    ])

    print("Number of classes:", len(classes))

    total_images = 0

    print("\nClass distribution:")

    for cls in classes:

        class_path = os.path.join(dataset_path, cls)

        images = [
            f for f in os.listdir(class_path)
            if f.lower().endswith(
                (".jpg", ".jpeg", ".png", ".bmp", ".webp")
            )
        ]

        total_images += len(images)

        print(f"{cls:<30} {len(images):>6}")

    print("-" * 70)
    print("Total images:", total_images)


Transfer_Data
Path: /kaggle/input/datasets/lizhecheng/mushroom-classification/Transfer data/Transfer data
Number of classes: 2

Class distribution:
Exidia                            435
Inocybe                           618
----------------------------------------------------------------------
Total images: 1053

Archive_Mushrooms
Path: /kaggle/input/datasets/lizhecheng/mushroom-classification/archive/Mushrooms
Number of classes: 9

Class distribution:
Agaricus                          353
Amanita                           750
Boletus                          1073
Cortinarius                       836
Entoloma                          364
Hygrocybe                         316
Lactarius                        1563
Russula                          1148
Suillus                           311
----------------------------------------------------------------------
Total images: 6714


In [4]:
from PIL import Image, ImageFile
import os

ImageFile.LOAD_TRUNCATED_IMAGES = False

BAD_IMAGES = []

DATASETS = {
    "Transfer_Data": "/kaggle/input/datasets/lizhecheng/mushroom-classification/Transfer data/Transfer data",
    "Archive_Mushrooms": "/kaggle/input/datasets/lizhecheng/mushroom-classification/archive/Mushrooms"
}

VALID_EXTENSIONS = (".jpg", ".jpeg", ".png", ".bmp", ".webp")

print("=" * 70)
print("CHECKING ALL IMAGES")
print("=" * 70)

total_checked = 0

for dataset_name, dataset_path in DATASETS.items():

    print(f"\nChecking: {dataset_name}")

    for class_name in sorted(os.listdir(dataset_path)):

        class_path = os.path.join(dataset_path, class_name)

        if not os.path.isdir(class_path):
            continue

        for filename in os.listdir(class_path):

            if not filename.lower().endswith(VALID_EXTENSIONS):
                continue

            image_path = os.path.join(class_path, filename)
            total_checked += 1

            try:
                # First check
                with Image.open(image_path) as img:
                    img.verify()

                # Second check: actually load the image
                with Image.open(image_path) as img:
                    img.load()

            except Exception as e:

                BAD_IMAGES.append({
                    "dataset": dataset_name,
                    "class": class_name,
                    "filename": filename,
                    "path": image_path,
                    "error": str(e)
                })

print("\n" + "=" * 70)
print("IMAGE CHECK COMPLETE")
print("=" * 70)

print("Total images checked :", total_checked)
print("Bad/corrupted images:", len(BAD_IMAGES))

if len(BAD_IMAGES) > 0:

    print("\nBAD IMAGES:")
    
    for item in BAD_IMAGES:
        print("\nDataset :", item["dataset"])
        print("Class   :", item["class"])
        print("File    :", item["filename"])
        print("Error   :", item["error"])

else:
    print("\n✅ No corrupted or truncated images found!")

CHECKING ALL IMAGES

Checking: Transfer_Data

Checking: Archive_Mushrooms

IMAGE CHECK COMPLETE
Total images checked : 7767
Bad/corrupted images: 1

BAD IMAGES:

Dataset : Archive_Mushrooms
Class   : Russula
File    : 092_43B354vYxm8.jpg
Error   : image file is truncated (92 bytes not processed)


In [5]:
import os
import shutil

CLEANED_PATH = "/kaggle/working/cleaned_mushroom_dataset"

# Remove old cleaned dataset if it already exists
if os.path.exists(CLEANED_PATH):
    shutil.rmtree(CLEANED_PATH)

os.makedirs(CLEANED_PATH, exist_ok=True)

copied = 0
skipped = 0

for dataset_name, dataset_path in DATASETS.items():

    for class_name in sorted(os.listdir(dataset_path)):

        class_path = os.path.join(dataset_path, class_name)

        if not os.path.isdir(class_path):
            continue

        # Create class folder
        output_class_path = os.path.join(
            CLEANED_PATH,
            class_name
        )

        os.makedirs(output_class_path, exist_ok=True)

        for filename in os.listdir(class_path):

            if not filename.lower().endswith(VALID_EXTENSIONS):
                continue

            source_path = os.path.join(class_path, filename)

            # Skip the corrupted image
            if (
                dataset_name == "Archive_Mushrooms"
                and class_name == "Russula"
                and filename == "092_43B354vYxm8.jpg"
            ):
                skipped += 1
                print("Skipped corrupted image:", source_path)
                continue

            destination_path = os.path.join(
                output_class_path,
                filename
            )

            shutil.copy2(source_path, destination_path)

            copied += 1

print("\n" + "=" * 70)
print("CLEAN DATASET CREATED")
print("=" * 70)

print("Images copied :", copied)
print("Images skipped:", skipped)
print("Location      :", CLEANED_PATH)

Skipped corrupted image: /kaggle/input/datasets/lizhecheng/mushroom-classification/archive/Mushrooms/Russula/092_43B354vYxm8.jpg

CLEAN DATASET CREATED
Images copied : 7766
Images skipped: 1
Location      : /kaggle/working/cleaned_mushroom_dataset


In [6]:
import os
import hashlib
from collections import defaultdict

CLEANED_PATH = "/kaggle/working/cleaned_mushroom_dataset"

image_hashes = defaultdict(list)

VALID_EXTENSIONS = (".jpg", ".jpeg", ".png", ".bmp", ".webp")

total_images = 0

print("=" * 70)
print("CHECKING FOR DUPLICATE IMAGES")
print("=" * 70)

for class_name in sorted(os.listdir(CLEANED_PATH)):

    class_path = os.path.join(CLEANED_PATH, class_name)

    if not os.path.isdir(class_path):
        continue

    for filename in os.listdir(class_path):

        if not filename.lower().endswith(VALID_EXTENSIONS):
            continue

        image_path = os.path.join(class_path, filename)

        # Calculate MD5 hash
        with open(image_path, "rb") as f:
            file_hash = hashlib.md5(f.read()).hexdigest()

        image_hashes[file_hash].append(
            (class_name, filename, image_path)
        )

        total_images += 1


# Find duplicates
duplicates = {
    h: files
    for h, files in image_hashes.items()
    if len(files) > 1
}

print("\nTotal images checked:", total_images)
print("Unique images:", len(image_hashes))
print("Duplicate groups:", len(duplicates))

duplicate_count = sum(
    len(files) - 1
    for files in duplicates.values()
)

print("Duplicate images:", duplicate_count)


if duplicates:

    print("\n" + "=" * 70)
    print("DUPLICATE GROUPS")
    print("=" * 70)

    for i, (hash_value, files) in enumerate(duplicates.items(), 1):

        print(f"\nDuplicate group {i}:")

        for class_name, filename, path in files:
            print(f"  {class_name} --> {filename}")

else:

    print("\n✅ No exact duplicate images found!")

CHECKING FOR DUPLICATE IMAGES

Total images checked: 7766
Unique images: 7692
Duplicate groups: 74
Duplicate images: 74

DUPLICATE GROUPS

Duplicate group 1:
  Agaricus --> 181_42srgvDitnE.jpg
  Agaricus --> 180_eZSp7CXrIps.jpg

Duplicate group 2:
  Agaricus --> 151_QKzgc9MO9LM.jpg
  Agaricus --> 152_uLPPbLUrAro.jpg

Duplicate group 3:
  Amanita --> 732__6NHENIp15Y.jpg
  Amanita --> 765_EloICHZR9ow.jpg

Duplicate group 4:
  Amanita --> 139_2tM1nrJmAeY.jpg
  Amanita --> 140_2tM1nrJmAeY.jpg

Duplicate group 5:
  Boletus --> 0051_rBIC-Uy9KzI.jpg
  Boletus --> 0105_7lVm0lLO6FQ.jpg

Duplicate group 6:
  Boletus --> 0757_vqAhKJVV4y0.jpg
  Boletus --> 0756_5lg4_v0qifc.jpg

Duplicate group 7:
  Cortinarius --> 445_FGpuhpGk3Ao.jpg
  Cortinarius --> 441_FZoHpOe6uOM.jpg

Duplicate group 8:
  Cortinarius --> 100_Opepcm1b-cs.jpg
  Cortinarius --> 101_c0c-rcWwaSo.jpg

Duplicate group 9:
  Exidia --> TAK2012PIC38700329.jpg
  Exidia --> TAK2012PIC66369143.jpg

Duplicate group 10:
  Exidia --> BWP2010P

In [7]:
import os
import shutil
import hashlib
from collections import defaultdict

SOURCE_PATH = "/kaggle/working/cleaned_mushroom_dataset"

UNIQUE_PATH = "/kaggle/working/mushroom_unique_dataset"

# Remove previous output if it exists
if os.path.exists(UNIQUE_PATH):
    shutil.rmtree(UNIQUE_PATH)

os.makedirs(UNIQUE_PATH, exist_ok=True)

VALID_EXTENSIONS = (".jpg", ".jpeg", ".png", ".bmp", ".webp")

seen_hashes = set()

copied = 0
duplicates_removed = 0

for class_name in sorted(os.listdir(SOURCE_PATH)):

    class_path = os.path.join(SOURCE_PATH, class_name)

    if not os.path.isdir(class_path):
        continue

    output_class_path = os.path.join(
        UNIQUE_PATH,
        class_name
    )

    os.makedirs(output_class_path, exist_ok=True)

    for filename in sorted(os.listdir(class_path)):

        if not filename.lower().endswith(VALID_EXTENSIONS):
            continue

        source_path = os.path.join(class_path, filename)

        # Calculate image/file hash
        with open(source_path, "rb") as f:
            file_hash = hashlib.md5(f.read()).hexdigest()

        # Duplicate found
        if file_hash in seen_hashes:
            duplicates_removed += 1
            continue

        # First occurrence → keep it
        seen_hashes.add(file_hash)

        destination_path = os.path.join(
            output_class_path,
            filename
        )

        shutil.copy2(source_path, destination_path)

        copied += 1

print("=" * 70)
print("DUPLICATE CLEANING COMPLETE")
print("=" * 70)

print("Unique images copied :", copied)
print("Duplicate copies removed:", duplicates_removed)
print("Output location:", UNIQUE_PATH)

DUPLICATE CLEANING COMPLETE
Unique images copied : 7692
Duplicate copies removed: 74
Output location: /kaggle/working/mushroom_unique_dataset


In [8]:
import os
from collections import Counter

UNIQUE_PATH = "/kaggle/working/mushroom_unique_dataset"

class_counts = {}

VALID_EXTENSIONS = (".jpg", ".jpeg", ".png", ".bmp", ".webp")

for class_name in sorted(os.listdir(UNIQUE_PATH)):

    class_path = os.path.join(UNIQUE_PATH, class_name)

    if not os.path.isdir(class_path):
        continue

    count = 0

    for filename in os.listdir(class_path):

        if filename.lower().endswith(VALID_EXTENSIONS):
            count += 1

    class_counts[class_name] = count


print("=" * 70)
print("FINAL CLASS DISTRIBUTION")
print("=" * 70)

for i, (class_name, count) in enumerate(class_counts.items()):

    print(f"{i:2d}. {class_name:<30} {count:>5}")

print("=" * 70)

total_images = sum(class_counts.values())

print("Number of classes :", len(class_counts))
print("Total images      :", total_images)
print("Largest class     :", max(class_counts.values()))
print("Smallest class    :", min(class_counts.values()))

largest_class = max(class_counts, key=class_counts.get)
smallest_class = min(class_counts, key=class_counts.get)

print("Largest class name :", largest_class)
print("Smallest class name:", smallest_class)

imbalance_ratio = max(class_counts.values()) / min(class_counts.values())

print(f"Imbalance ratio    : {imbalance_ratio:.2f}x")

FINAL CLASS DISTRIBUTION
 0. Agaricus                         351
 1. Amanita                          748
 2. Boletus                         1071
 3. Cortinarius                      834
 4. Entoloma                         364
 5. Exidia                           433
 6. Hygrocybe                        315
 7. Inocybe                          615
 8. Lactarius                       1505
 9. Russula                         1145
10. Suillus                          311
Number of classes : 11
Total images      : 7692
Largest class     : 1505
Smallest class    : 311
Largest class name : Lactarius
Smallest class name: Suillus
Imbalance ratio    : 4.84x


In [9]:
import os
import shutil
import random
from collections import defaultdict

SOURCE_PATH = "/kaggle/working/mushroom_unique_dataset"
SPLIT_PATH = "/kaggle/working/mushroom_split_dataset"

# Reproducibility
random.seed(42)

# Remove previous split if it exists
if os.path.exists(SPLIT_PATH):
    shutil.rmtree(SPLIT_PATH)

# Create folders
for split in ["train", "val", "test"]:
    os.makedirs(os.path.join(SPLIT_PATH, split), exist_ok=True)


TRAIN_RATIO = 0.70
VAL_RATIO = 0.15
TEST_RATIO = 0.15

VALID_EXTENSIONS = (".jpg", ".jpeg", ".png", ".bmp", ".webp")

total_train = 0
total_val = 0
total_test = 0

print("=" * 70)
print("CREATING STRATIFIED DATASET SPLIT")
print("=" * 70)

for class_name in sorted(os.listdir(SOURCE_PATH)):

    class_path = os.path.join(SOURCE_PATH, class_name)

    if not os.path.isdir(class_path):
        continue

    images = [
        f for f in os.listdir(class_path)
        if f.lower().endswith(VALID_EXTENSIONS)
    ]

    # Shuffle deterministically
    random.shuffle(images)

    total = len(images)

    train_end = int(total * TRAIN_RATIO)
    val_end = train_end + int(total * VAL_RATIO)

    train_images = images[:train_end]
    val_images = images[train_end:val_end]
    test_images = images[val_end:]

    # Create class folders
    for split in ["train", "val", "test"]:
        os.makedirs(
            os.path.join(SPLIT_PATH, split, class_name),
            exist_ok=True
        )

    # Copy images
    for filename in train_images:
        shutil.copy2(
            os.path.join(class_path, filename),
            os.path.join(SPLIT_PATH, "train", class_name, filename)
        )

    for filename in val_images:
        shutil.copy2(
            os.path.join(class_path, filename),
            os.path.join(SPLIT_PATH, "val", class_name, filename)
        )

    for filename in test_images:
        shutil.copy2(
            os.path.join(class_path, filename),
            os.path.join(SPLIT_PATH, "test", class_name, filename)
        )

    total_train += len(train_images)
    total_val += len(val_images)
    total_test += len(test_images)

    print(
        f"{class_name:<30}"
        f"Train: {len(train_images):>4}   "
        f"Val: {len(val_images):>4}   "
        f"Test: {len(test_images):>4}"
    )

print("=" * 70)
print("SPLIT COMPLETE")
print("=" * 70)

print("Total train:", total_train)
print("Total val  :", total_val)
print("Total test :", total_test)
print("Total      :", total_train + total_val + total_test)

print("\nDataset location:")
print(SPLIT_PATH)

CREATING STRATIFIED DATASET SPLIT
Agaricus                      Train:  245   Val:   52   Test:   54
Amanita                       Train:  523   Val:  112   Test:  113
Boletus                       Train:  749   Val:  160   Test:  162
Cortinarius                   Train:  583   Val:  125   Test:  126
Entoloma                      Train:  254   Val:   54   Test:   56
Exidia                        Train:  303   Val:   64   Test:   66
Hygrocybe                     Train:  220   Val:   47   Test:   48
Inocybe                       Train:  430   Val:   92   Test:   93
Lactarius                     Train: 1053   Val:  225   Test:  227
Russula                       Train:  801   Val:  171   Test:  173
Suillus                       Train:  217   Val:   46   Test:   48
SPLIT COMPLETE
Total train: 5378
Total val  : 1148
Total test : 1166
Total      : 7692

Dataset location:
/kaggle/working/mushroom_split_dataset


In [10]:
import os

SPLIT_PATH = "/kaggle/working/mushroom_split_dataset"

print("=" * 70)
print("DATASET SPLIT VERIFICATION")
print("=" * 70)

total = 0

for split in ["train", "val", "test"]:

    split_path = os.path.join(SPLIT_PATH, split)

    split_total = 0

    print(f"\n{split.upper()}")

    for class_name in sorted(os.listdir(split_path)):

        class_path = os.path.join(split_path, class_name)

        if not os.path.isdir(class_path):
            continue

        count = len([
            f for f in os.listdir(class_path)
            if f.lower().endswith(
                (".jpg", ".jpeg", ".png", ".bmp", ".webp")
            )
        ])

        split_total += count

        print(f"{class_name:<30} {count:>5}")

    print("Total:", split_total)

    total += split_total

print("\n" + "=" * 70)
print("TOTAL IMAGES:", total)
print("=" * 70)

DATASET SPLIT VERIFICATION

TRAIN
Agaricus                         245
Amanita                          523
Boletus                          749
Cortinarius                      583
Entoloma                         254
Exidia                           303
Hygrocybe                        220
Inocybe                          430
Lactarius                       1053
Russula                          801
Suillus                          217
Total: 5378

VAL
Agaricus                          52
Amanita                          112
Boletus                          160
Cortinarius                      125
Entoloma                          54
Exidia                            64
Hygrocybe                         47
Inocybe                           92
Lactarius                        225
Russula                          171
Suillus                           46
Total: 1148

TEST
Agaricus                          54
Amanita                          113
Boletus                          162
Cortin

In [11]:
import torch
import torchvision
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

# ============================================================
# PATHS
# ============================================================

SPLIT_PATH = "/kaggle/working/mushroom_split_dataset"

TRAIN_PATH = f"{SPLIT_PATH}/train"
VAL_PATH   = f"{SPLIT_PATH}/val"
TEST_PATH  = f"{SPLIT_PATH}/test"


# ============================================================
# DEVICE
# ============================================================

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("PyTorch version :", torch.__version__)
print("Torchvision     :", torchvision.__version__)
print("Device          :", device)

if torch.cuda.is_available():
    print("GPU             :", torch.cuda.get_device_name(0))


# ============================================================
# IMAGE SIZE
# ============================================================

IMAGE_SIZE = 224


# ============================================================
# IMAGENET NORMALIZATION
# ============================================================

IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]


# ============================================================
# TRAIN TRANSFORMS
# ============================================================

train_transform = transforms.Compose([

    transforms.RandomResizedCrop(
        IMAGE_SIZE,
        scale=(0.7, 1.0),
        ratio=(0.75, 1.33)
    ),

    transforms.RandomHorizontalFlip(p=0.5),

    transforms.RandomVerticalFlip(p=0.2),

    transforms.RandomRotation(15),

    transforms.ColorJitter(
        brightness=0.2,
        contrast=0.2,
        saturation=0.2,
        hue=0.05
    ),

    transforms.ToTensor(),

    transforms.Normalize(
        IMAGENET_MEAN,
        IMAGENET_STD
    ),

    transforms.RandomErasing(
        p=0.25,
        scale=(0.02, 0.15),
        ratio=(0.3, 3.3)
    )
])


# ============================================================
# VALIDATION / TEST TRANSFORMS
# ============================================================

eval_transform = transforms.Compose([

    transforms.Resize(
        int(IMAGE_SIZE * 1.14)
    ),

    transforms.CenterCrop(
        IMAGE_SIZE
    ),

    transforms.ToTensor(),

    transforms.Normalize(
        IMAGENET_MEAN,
        IMAGENET_STD
    )
])


# ============================================================
# DATASETS
# ============================================================

train_dataset = datasets.ImageFolder(
    TRAIN_PATH,
    transform=train_transform
)

val_dataset = datasets.ImageFolder(
    VAL_PATH,
    transform=eval_transform
)

test_dataset = datasets.ImageFolder(
    TEST_PATH,
    transform=eval_transform
)


# ============================================================
# DATALOADERS
# ============================================================

BATCH_SIZE = 32

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=2,
    pin_memory=True,
    persistent_workers=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    pin_memory=True,
    persistent_workers=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    pin_memory=True,
    persistent_workers=True
)


# ============================================================
# INFORMATION
# ============================================================

print("\n" + "=" * 70)
print("DATASET INFORMATION")
print("=" * 70)

print("Number of classes :", len(train_dataset.classes))
print("Classes           :", train_dataset.classes)

print("\nTraining images   :", len(train_dataset))
print("Validation images :", len(val_dataset))
print("Test images       :", len(test_dataset))

print("\nBatch size        :", BATCH_SIZE)
print("Image size        :", IMAGE_SIZE)

PyTorch version : 2.11.0+cu128
Torchvision     : 0.26.0+cu128
Device          : cuda
GPU             : Tesla T4

DATASET INFORMATION
Number of classes : 11
Classes           : ['Agaricus', 'Amanita', 'Boletus', 'Cortinarius', 'Entoloma', 'Exidia', 'Hygrocybe', 'Inocybe', 'Lactarius', 'Russula', 'Suillus']

Training images   : 5378
Validation images : 1148
Test images       : 1166

Batch size        : 32
Image size        : 224


In [12]:
import torch
from collections import Counter

# Get training labels
train_labels = train_dataset.targets

# Count images in each class
class_counts = Counter(train_labels)

print("Class distribution:")
for class_idx, count in sorted(class_counts.items()):
    print(f"{train_dataset.classes[class_idx]:30s}: {count}")

# Calculate class weights
num_classes = len(train_dataset.classes)
total_samples = len(train_labels)

class_weights = []

for i in range(num_classes):
    count = class_counts[i]
    weight = total_samples / (num_classes * count)
    class_weights.append(weight)

class_weights = torch.tensor(
    class_weights,
    dtype=torch.float32
).to(device)

print("\nClass weights:")
for i, weight in enumerate(class_weights):
    print(f"{train_dataset.classes[i]:30s}: {weight.item():.4f}")

Class distribution:
Agaricus                      : 245
Amanita                       : 523
Boletus                       : 749
Cortinarius                   : 583
Entoloma                      : 254
Exidia                        : 303
Hygrocybe                     : 220
Inocybe                       : 430
Lactarius                     : 1053
Russula                       : 801
Suillus                       : 217

Class weights:
Agaricus                      : 1.9955
Amanita                       : 0.9348
Boletus                       : 0.6527
Cortinarius                   : 0.8386
Entoloma                      : 1.9248
Exidia                        : 1.6136
Hygrocybe                     : 2.2223
Inocybe                       : 1.1370
Lactarius                     : 0.4643
Russula                       : 0.6104
Suillus                       : 2.2530


In [13]:
import torch
import torch.nn as nn
from torchvision.models import efficientnet_v2_s
from torchvision.models import EfficientNet_V2_S_Weights

# Load pretrained EfficientNetV2-S
weights = EfficientNet_V2_S_Weights.DEFAULT

model = efficientnet_v2_s(weights=weights)

# Number of classes
num_classes = len(train_dataset.classes)

# Replace final classifier
in_features = model.classifier[1].in_features

model.classifier[1] = nn.Linear(
    in_features,
    num_classes
)

# Move model to GPU
model = model.to(device)

print("Model: EfficientNetV2-S")
print("Number of classes:", num_classes)
print("Device:", device)
print("Classifier input features:", in_features)

Downloading: "https://download.pytorch.org/models/efficientnet_v2_s-dd5fe13b.pth" to /root/.cache/torch/hub/checkpoints/efficientnet_v2_s-dd5fe13b.pth


100%|██████████| 82.7M/82.7M [00:00<00:00, 245MB/s]


Model: EfficientNetV2-S
Number of classes: 11
Device: cuda
Classifier input features: 1280


In [14]:
import torch

print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("CUDA version:", torch.version.cuda)
else:
    print("⚠️ GPU is NOT available")

PyTorch version: 2.11.0+cu128
CUDA available: True
GPU: Tesla T4
CUDA version: 12.8


In [15]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Device:", device)

Device: cuda


In [16]:
import torch.nn as nn

criterion = nn.CrossEntropyLoss(
    weight=class_weights,
    label_smoothing=0.1
)

print("Loss function: Weighted CrossEntropyLoss")
print("Label smoothing: 0.1")

Loss function: Weighted CrossEntropyLoss
Label smoothing: 0.1


In [17]:
import torch.optim as optim

optimizer = optim.AdamW(
    [
        {
            "params": model.features.parameters(),
            "lr": 1e-5
        },
        {
            "params": model.classifier.parameters(),
            "lr": 1e-4
        }
    ],
    weight_decay=1e-4
)

print("Optimizer: AdamW")
print("Backbone LR: 1e-5")
print("Classifier LR: 1e-4")
print("Weight decay: 1e-4")

Optimizer: AdamW
Backbone LR: 1e-5
Classifier LR: 1e-4
Weight decay: 1e-4


In [18]:
EPOCHS = 30

scheduler = optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max=EPOCHS,
    eta_min=1e-7
)

print("Epochs:", EPOCHS)
print("Scheduler: CosineAnnealingLR")
print("Minimum LR: 1e-7")

Epochs: 30
Scheduler: CosineAnnealingLR
Minimum LR: 1e-7


In [19]:
scaler = torch.amp.GradScaler("cuda")

print("Mixed Precision: Enabled")

Mixed Precision: Enabled


In [20]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from PIL import ImageFile
import torch
import os

ImageFile.LOAD_TRUNCATED_IMAGES = True

best_val_accuracy = 0.0
best_epoch = 0
best_val_precision = 0.0
best_val_recall = 0.0
best_val_f1 = 0.0

best_model_path = "/kaggle/working/best_efficientnetv2_s.pth"

for epoch in range(EPOCHS):

    # =========================================================
    # TRAINING
    # =========================================================
    model.train()

    train_loss = 0.0
    train_predictions = []
    train_labels = []

    for images, labels in train_loader:

        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)

        with torch.amp.autocast(device_type="cuda"):
            outputs = model(images)
            loss = criterion(outputs, labels)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        train_loss += loss.item() * images.size(0)

        predictions = torch.argmax(outputs, dim=1)

        train_predictions.extend(predictions.detach().cpu().numpy())
        train_labels.extend(labels.detach().cpu().numpy())

    train_loss /= len(train_dataset)

    train_accuracy = accuracy_score(
        train_labels,
        train_predictions
    )

    train_precision = precision_score(
        train_labels,
        train_predictions,
        average="weighted",
        zero_division=0
    )

    train_recall = recall_score(
        train_labels,
        train_predictions,
        average="weighted",
        zero_division=0
    )

    train_f1 = f1_score(
        train_labels,
        train_predictions,
        average="weighted",
        zero_division=0
    )

    # =========================================================
    # VALIDATION
    # =========================================================
    model.eval()

    val_loss = 0.0
    val_predictions = []
    val_labels = []

    with torch.no_grad():

        for images, labels in val_loader:

            images = images.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)

            with torch.amp.autocast(device_type="cuda"):
                outputs = model(images)
                loss = criterion(outputs, labels)

            val_loss += loss.item() * images.size(0)

            predictions = torch.argmax(outputs, dim=1)

            val_predictions.extend(
                predictions.cpu().numpy()
            )

            val_labels.extend(
                labels.cpu().numpy()
            )

    val_loss /= len(val_dataset)

    val_accuracy = accuracy_score(
        val_labels,
        val_predictions
    )

    val_precision = precision_score(
        val_labels,
        val_predictions,
        average="weighted",
        zero_division=0
    )

    val_recall = recall_score(
        val_labels,
        val_predictions,
        average="weighted",
        zero_division=0
    )

    val_f1 = f1_score(
        val_labels,
        val_predictions,
        average="weighted",
        zero_division=0
    )

    # =========================================================
    # LEARNING RATE UPDATE
    # =========================================================
    scheduler.step()

    current_lr = optimizer.param_groups[0]["lr"]

    # =========================================================
    # SAVE BEST MODEL
    # =========================================================
    if val_accuracy > best_val_accuracy:

        best_val_accuracy = val_accuracy
        best_epoch = epoch + 1

        best_val_precision = val_precision
        best_val_recall = val_recall
        best_val_f1 = val_f1

        torch.save(
            {
                "epoch": epoch + 1,
                "model_state_dict": model.state_dict(),
                "optimizer_state_dict": optimizer.state_dict(),
                "val_accuracy": val_accuracy,
                "val_precision": val_precision,
                "val_recall": val_recall,
                "val_f1": val_f1,
                "classes": train_dataset.classes
            },
            best_model_path
        )

        best_marker = " ⭐ BEST"

    else:
        best_marker = ""

    # =========================================================
    # PRINT RESULTS
    # =========================================================
    print(
        f"\nEpoch [{epoch+1:02d}/{EPOCHS}]"
        f"{best_marker}"
    )

    print(
        f"Train Loss      : {train_loss:.4f}"
    )

    print(
        f"Train Accuracy  : {train_accuracy*100:.2f}%"
    )

    print(
        f"Train Precision : {train_precision*100:.2f}%"
    )

    print(
        f"Train Recall    : {train_recall*100:.2f}%"
    )

    print(
        f"Train F1        : {train_f1*100:.2f}%"
    )

    print(
        f"Val Loss        : {val_loss:.4f}"
    )

    print(
        f"Val Accuracy    : {val_accuracy*100:.2f}%"
    )

    print(
        f"Val Precision   : {val_precision*100:.2f}%"
    )

    print(
        f"Val Recall      : {val_recall*100:.2f}%"
    )

    print(
        f"Val F1          : {val_f1*100:.2f}%"
    )

    print(
        f"Learning Rate   : {current_lr:.2e}"
    )


# =============================================================
# FINAL BEST RESULT
# =============================================================

print("\n" + "=" * 60)
print("BEST VALIDATION RESULT")
print("=" * 60)

print(
    f"Best Epoch       : {best_epoch}"
)

print(
    f"Best Val Accuracy: {best_val_accuracy*100:.2f}%"
)

print(
    f"Val Precision    : {best_val_precision*100:.2f}%"
)

print(
    f"Val Recall       : {best_val_recall*100:.2f}%"
)

print(
    f"Val F1           : {best_val_f1*100:.2f}%"
)

print(
    f"Best Model Saved : {best_model_path}"
)

print("=" * 60)


Epoch [01/30] ⭐ BEST
Train Loss      : 2.3586
Train Accuracy  : 23.41%
Train Precision : 28.57%
Train Recall    : 23.41%
Train F1        : 23.76%
Val Loss        : 2.2437
Val Accuracy    : 47.74%
Val Precision   : 50.44%
Val Recall      : 47.74%
Val F1          : 45.92%
Learning Rate   : 9.97e-06

Epoch [02/30] ⭐ BEST
Train Loss      : 1.9427
Train Accuracy  : 47.47%
Train Precision : 52.00%
Train Recall    : 47.47%
Train F1        : 46.67%
Val Loss        : 1.8422
Val Accuracy    : 56.10%
Val Precision   : 59.60%
Val Recall      : 56.10%
Val F1          : 53.11%
Learning Rate   : 9.89e-06

Epoch [03/30] ⭐ BEST
Train Loss      : 1.6072
Train Accuracy  : 58.68%
Train Precision : 63.39%
Train Recall    : 58.68%
Train F1        : 58.66%
Val Loss        : 1.6306
Val Accuracy    : 65.24%
Val Precision   : 66.82%
Val Recall      : 65.24%
Val F1          : 64.04%
Learning Rate   : 9.76e-06

Epoch [04/30] ⭐ BEST
Train Loss      : 1.4208
Train Accuracy  : 65.60%
Train Precision : 68.63%
Train 